# Feature generation for one polymer–solvent data point

This notebook is a simplified, single-data-point version of the workflow used to generate the 77 machine-learning features reported in the Supporting Information.

## Requirements

- Python 3
- `numpy`
- `pandas`
- `rdkit`
- `fastsolv`

The FASTSOLV package must have access to its pretrained checkpoints. The COSMO-RS solvation free energy is supplied directly as an input because it was calculated in a separate workflow.

Edit only the input cells below, then run all cells in order. The last cell prints the features used by the machine-learning models.

**Notes**

- Temperature is entered in °C and converted to K.
- Concentration is entered directly in mg/mL; concentration-range expansion is not used.
- A second solvent and a second side-chain SMILES are optional.

In [1]:
# ============================================================
# 1. Imports and fixed model metadata
# ============================================================

import re
import numpy as np
import pandas as pd

from rdkit import Chem
from rdkit.Chem import AllChem, Crippen, Descriptors, Lipinski, rdMolDescriptors

FAMILY_CATEGORIES = [
    "Thiophene",
    "DOT",
    "Qx",
    "II",
    "DPP",
    "TPT",
    "NDI",
    "BDT",
    "Polyfluorene",
    "IDT",
    "Ladder",
]

FINAL_FEATURE_COLUMNS = [
    "MW_Mn", "MW_Mw", "PDI", "MolWt_monomer", "Temperature",
    "R_atoms_C", "R_atom_all", "R_bond_1", "R_bond_2",
    "R_ring_1", "R_ring_2", "R_ring_3", "R_ring_4", "R_ring_5", "R_ring_6",
    "SC_n_branch_points_Norm_MolWt",
    "SC_atoms_after_branching_CO_Norm_MolWt",
    "SC_n_units_Norm_MolWt",
    "SC_atoms_before_branching_CO_Norm_MolWt",
    "SC_atoms_before_plus_after_CO_Norm_MolWt",
    "SC_C_atoms_Norm_MolWt", "SC_O_atoms_Norm_MolWt",
    "SC_total_atoms_CO_Norm_MolWt", "SC_bonds_CC_or_CO_Norm_MolWt",
    "SC_total_branches_Norm_MolWt",
    "BB_C_atoms_Norm_MolWt", "BB_hetero_atoms_Norm_MolWt",
    "BB_CC_rotatable_bonds_Norm_MolWt", "BB_CeqC_bonds_Norm_MolWt",
    "BB_rings_total_Norm_MolWt", "BB_conjugated_units_Norm_MolWt",
    "Monomer_NumHDonors_Norm_MolWt", "Monomer_NumHAcceptors_Norm_MolWt",
    "Monomer_MolLogP", "Monomer_NumSpiroAtoms_Norm_MolWt",
    "Monomer_NumSaturatedRings_Norm_MolWt",
    "Monomer_NumAliphaticRings_Norm_MolWt",
    "Monomer_NumAromaticRings_Norm_MolWt",
    "Monomer_NumBridgeheadAtoms_Norm_MolWt",
    "Backbone_MolLogP", "SideChain_MolLogP", "BBminusSC_MolLogP",
    "Solvent_MolLogP", "Solvent_NumHDonors", "Solvent_NumHAcceptors",
    "Monomer_Asphericity", "Monomer_InertialShapeFactor", "Monomer_Eccentricity", "Monomer_TPSA",
    "Backbone_Asphericity", "Backbone_InertialShapeFactor", "Backbone_Eccentricity", "Backbone_TPSA",
    "SideChain_Asphericity", "SideChain_InertialShapeFactor", "SideChain_Eccentricity", "SideChain_TPSA",
    "Solvent_Asphericity", "Solvent_InertialShapeFactor", "Solvent_Eccentricity", "Solvent_TPSA",
    "Monomer_fastsolv", "Backbone_fastsolv", "SideChain_fastsolv",
    "family_Thiophene", "family_DOT", "family_Qx", "family_II", "family_DPP", "family_TPT",
    "family_NDI", "family_BDT", "family_Polyfluorene", "family_IDT", "family_Ladder",
    "dG_solv_cosmo", "Concentration_mg_mL",
]

assert len(FINAL_FEATURE_COLUMNS) == 77

## Input 1 — identity, conditions, structures, and solvent information

In [2]:
# ============================================================
# 2. USER INPUT: identity, conditions, and molecular structures
# ============================================================
# Replace the illustrative values below with one experimental data point.

polymer_name = "AcDOT-[C10]2-co-[OE3]2"
family = "Thiophene"

temperature_C = 25.0
concentration_mg_mL = 100.0

MW_Mn = 30000.0
MW_Mw = 52500.0
PDI = 1.75

# Dummy atoms (*) in the repeat-unit SMILES are removed before
# calculating RDKit and FASTSOLV descriptors.
monomer_smiles = "*c2sc(c1sc(*)c(OCCOCCOCCOC)c1OCCOCCOCCOC)c(OCCCCCCCCCC)c2OCCCCCCCCCC"
backbone_smiles = "COc2csc(c1scc(OC)c1OC)c2OC"
sidechain1_smiles = "COCCOCCOC"
sidechain2_smiles = "CCCCCCCCC"  # optional

solvent_name = "Chloroform"
solvent1_smiles = "ClC(Cl)Cl"
solvent1_ratio = 1.0
solvent2_smiles = None    # optional
solvent2_ratio = 0.0

# Supplied from the separate COSMO-RS workflow.
dG_solv_cosmo = -45.062557

# g-factor = -0.08

# Ref: 10.1021/10.1021/jacs.5c07995

## Input 2 — manually curated side-chain and backbone counts

In [3]:
# ============================================================
# 3. USER INPUT: SC_ and BB_ quantities
# ============================================================

SC_C_atoms_input = 34
SC_O_atoms_input = 10
SC_bonds_CC_or_CO_input = 40
SC_n_branch_points_input = 0
SC_atoms_before_branching_CO_input = 0
SC_atoms_after_branching_CO_input = 44
SC_total_branches_input = 4
SC_n_units_input = 4

BB_C_atoms_input = 8
BB_hetero_atoms_input = 2
BB_CC_rotatable_bonds_input = 2
BB_CeqC_bonds_input = 4
BB_rings_total_input = 2
BB_conjugated_units_input = 2

In [4]:
# ============================================================
# 4. Validate inputs and construct the one-row raw table
# ============================================================

if family not in FAMILY_CATEGORIES:
    raise ValueError(f"Unknown family: {family!r}. Choose one of: {FAMILY_CATEGORIES}")

required_positive = {
    "MW_Mn": MW_Mn,
    "MW_Mw": MW_Mw,
    "PDI": PDI,
    "concentration_mg_mL": concentration_mg_mL,
    "solvent1_ratio": solvent1_ratio,
}
for name, value in required_positive.items():
    if value is None or not np.isfinite(float(value)) or float(value) <= 0:
        raise ValueError(f"{name} must be a positive finite number.")

if solvent2_smiles is None:
    solvent2_ratio = 0.0
elif float(solvent2_ratio) <= 0:
    raise ValueError("solvent2_ratio must be positive when solvent2_smiles is provided.")

for name, smi in {
    "monomer_smiles": monomer_smiles,
    "backbone_smiles": backbone_smiles,
    "sidechain1_smiles": sidechain1_smiles,
    "solvent1_smiles": solvent1_smiles,
}.items():
    if smi is None or not str(smi).strip():
        raise ValueError(f"{name} is required.")

df_raw = pd.DataFrame([{
    "Family": family,
    "Polymer name": polymer_name,
    "Polymer SMILES": monomer_smiles,
    "Polymer Backbone SMILES": backbone_smiles,
    "Polymer Side-Chain 1 SMILES": sidechain1_smiles,
    "Polymer Side-Chain 2 SMILES": sidechain2_smiles,
    "Solvent": solvent_name,
    "Solvent1 SMILES": solvent1_smiles,
    "Solvent1 ratio": float(solvent1_ratio),
    "Solvent2 SMILES": solvent2_smiles,
    "Solvent2 ratio": float(solvent2_ratio),
    "Temperature": float(temperature_C),
    "Concentration": float(concentration_mg_mL),
    "Molecular weight | Mn": float(MW_Mn),
    "Molecular weight | Mw": float(MW_Mw),
    "Molecular weight | Mw/Mn": float(PDI),
    "Side Chains (SC) | # C atoms on SC": float(SC_C_atoms_input),
    "Side Chains (SC) | # O atoms on SC": float(SC_O_atoms_input),
    "Side Chains (SC) | # C-C (or C-O )bond on SC": float(SC_bonds_CC_or_CO_input),
    "Side Chains (SC) | # SC braching pts": float(SC_n_branch_points_input),
    "Side Chains (SC) | # atoms (C and O) between braching and BB": float(SC_atoms_before_branching_CO_input),
    "Side Chains (SC) | # atoms (C and O) after branching": float(SC_atoms_after_branching_CO_input),
    "Side Chains (SC) | # total branches": float(SC_total_branches_input),
    "Side Chains (SC) | # SC  units": float(SC_n_units_input),
    "Backbones (BB) | # of C atoms on BB": float(BB_C_atoms_input),
    "Backbones (BB) | # of heteroatoms atoms on BB": float(BB_hetero_atoms_input),
    "Backbones (BB) | # of C-C rotatable on BB": float(BB_CC_rotatable_bonds_input),
    "Backbones (BB) | # of C=C bond on BB": float(BB_CeqC_bonds_input),
    "Backbones (BB) | # of rings": float(BB_rings_total_input),
    "Backbones (BB) | # of conjugated units on BB": float(BB_conjugated_units_input),
}])

df_feature_v1 = pd.DataFrame(index=df_raw.index)

print("Single input row created.")
display(df_raw.T)

Single input row created.


,0
Family,Thiophene
Polymer name,AcDOT-[C10]2-co-[OE3]2
Polymer SMILES,*c2sc(c1sc(*)c(OCCOCCOCCOC)c1OCCOCCOCCOC)c(OCC...
Polymer Backbone SMILES,COc2csc(c1scc(OC)c1OC)c2OC
Polymer Side-Chain 1 SMILES,COCCOCCOC
Polymer Side-Chain 2 SMILES,CCCCCCCCC
Solvent,Chloroform
Solvent1 SMILES,ClC(Cl)Cl
Solvent1 ratio,1.0
Solvent2 SMILES,None


In [5]:
# ============================================================
# 5. Molecular weight, temperature, and concentration
# ============================================================

def mol_without_dummies_and_sanitize(mol):
    """
    Remove dummy atoms (atomic number 0) and sanitize the molecule.

    This cleaned molecule is used for RDKit descriptors, logP,
    3D descriptors, and FASTSOLV inputs.

    Do NOT use this cleaned molecule to calculate repeat-unit
    molecular weight, because removing each terminal dummy atom
    causes RDKit to add an implicit hydrogen at that position.
    """
    if mol is None:
        return None

    dummy_indices = [
        atom.GetIdx()
        for atom in mol.GetAtoms()
        if atom.GetAtomicNum() == 0
    ]

    if dummy_indices:
        editable = Chem.RWMol(mol)

        for atom_index in sorted(dummy_indices, reverse=True):
            editable.RemoveAtom(atom_index)

        mol = editable.GetMol()

    try:
        Chem.SanitizeMol(mol)
        Chem.GetSymmSSSR(mol)
    except Exception:
        try:
            mol.UpdatePropertyCache(strict=False)
            Chem.GetSymmSSSR(mol)
        except Exception:
            return None

    return mol


def mol_from_smiles_clean(smiles):
    """
    Parse a SMILES and remove dummy atoms.

    Used for molecular descriptors and FASTSOLV, but not for
    repeat-unit molecular-weight calculation.
    """
    if smiles is None or pd.isna(smiles) or not str(smiles).strip():
        return None

    mol = Chem.MolFromSmiles(str(smiles).strip())

    return mol_without_dummies_and_sanitize(mol)


def clean_smiles_remove_dummy(smiles):
    mol = mol_from_smiles_clean(smiles)

    if mol is None:
        return np.nan

    return Chem.MolToSmiles(mol, canonical=True)


def molwt_from_repeat_unit_smiles(smiles):
    """
    Calculate the molecular weight of the polymer repeat unit.

    The original SMILES containing dummy atoms (*) is used directly.
    RDKit assigns zero mass to dummy atoms, while the bonds to the
    dummy atoms prevent implicit hydrogens from being added at the
    polymerization sites.

    Therefore, this gives the repeat-unit molecular weight rather
    than the hydrogen-capped molecular weight.
    """
    if smiles is None or pd.isna(smiles) or not str(smiles).strip():
        return np.nan

    mol = Chem.MolFromSmiles(str(smiles).strip())

    if mol is None:
        return np.nan

    return float(Descriptors.MolWt(mol))


df_feature_v1["MW_Mn"] = pd.to_numeric(
    df_raw["Molecular weight | Mn"],
    errors="coerce",
)

df_feature_v1["MW_Mw"] = pd.to_numeric(
    df_raw["Molecular weight | Mw"],
    errors="coerce",
)

df_feature_v1["PDI"] = pd.to_numeric(
    df_raw["Molecular weight | Mw/Mn"],
    errors="coerce",
)

# Important:
# Calculate repeat-unit MW from the original SMILES containing *.
df_feature_v1["MolWt_monomer"] = df_raw["Polymer SMILES"].apply(
    molwt_from_repeat_unit_smiles
)

df_feature_v1["Temperature"] = (
    pd.to_numeric(df_raw["Temperature"], errors="coerce") + 273.15
)

df_feature_v1["Concentration_mg_mL"] = pd.to_numeric(
    df_raw["Concentration"],
    errors="coerce",
)

if df_feature_v1["MolWt_monomer"].isna().any():
    raise ValueError(
        "Failed to calculate MolWt_monomer from monomer_smiles."
    )


# Optional diagnostic: compare repeat-unit MW with the H-capped MW
original_mol = Chem.MolFromSmiles(monomer_smiles)
cleaned_mol = mol_from_smiles_clean(monomer_smiles)

n_dummy_atoms = (
    sum(atom.GetAtomicNum() == 0 for atom in original_mol.GetAtoms())
    if original_mol is not None
    else 0
)

repeat_unit_mw = df_feature_v1.loc[0, "MolWt_monomer"]

hydrogen_capped_mw = (
    float(Descriptors.MolWt(cleaned_mol))
    if cleaned_mol is not None
    else np.nan
)

print(f"Number of dummy atoms: {n_dummy_atoms}")
print(f"Repeat-unit molecular weight: {repeat_unit_mw:.6f}")
print(f"H-capped molecular weight after removing *: {hydrogen_capped_mw:.6f}")
print(
    "Difference caused by added terminal hydrogens: "
    f"{hydrogen_capped_mw - repeat_unit_mw:.6f}"
)

Number of dummy atoms: 2
Repeat-unit molecular weight: 801.162000
H-capped molecular weight after removing *: 803.178000
Difference caused by added terminal hydrogens: 2.016000


In [6]:
# ============================================================
# 6. Side-chain/backbone ratios and manually curated features
# ============================================================

def safe_divide(numerator, denominator):
    numerator = float(numerator)
    denominator = float(denominator)
    return np.nan if denominator == 0 else numerator / denominator

sc_C = float(SC_C_atoms_input)
sc_O = float(SC_O_atoms_input)
sc_bonds = float(SC_bonds_CC_or_CO_input)
sc_units = float(SC_n_units_input)
sc_branches = float(SC_total_branches_input)

bb_C = float(BB_C_atoms_input)
bb_hetero = float(BB_hetero_atoms_input)
bb_rotCC = float(BB_CC_rotatable_bonds_input)
bb_CeqC = float(BB_CeqC_bonds_input)
bb_rings = float(BB_rings_total_input)
bb_conj = float(BB_conjugated_units_input)

df_feature_v1["R_atoms_C"] = safe_divide(sc_C, bb_C)
df_feature_v1["R_atom_all"] = safe_divide(sc_C + sc_O, bb_C + bb_hetero)
df_feature_v1["R_bond_1"] = safe_divide(sc_bonds, bb_CeqC)
df_feature_v1["R_bond_2"] = safe_divide(sc_bonds, bb_rotCC)
df_feature_v1["R_ring_1"] = safe_divide(sc_units, bb_rings)
df_feature_v1["R_ring_2"] = safe_divide(sc_branches, bb_rings)
df_feature_v1["R_ring_3"] = safe_divide(sc_bonds, bb_rings)
df_feature_v1["R_ring_4"] = safe_divide(sc_units, bb_conj)
df_feature_v1["R_ring_5"] = safe_divide(sc_branches, bb_conj)
df_feature_v1["R_ring_6"] = safe_divide(sc_bonds, bb_conj)

df_feature_v1["SC_n_branch_points"] = float(SC_n_branch_points_input)
df_feature_v1["SC_atoms_after_branching_CO"] = float(SC_atoms_after_branching_CO_input)
df_feature_v1["SC_n_units"] = sc_units
df_feature_v1["SC_atoms_before_branching_CO"] = float(SC_atoms_before_branching_CO_input)
df_feature_v1["SC_atoms_before_plus_after_CO"] = float(SC_atoms_before_branching_CO_input) + float(SC_atoms_after_branching_CO_input)
df_feature_v1["SC_C_atoms"] = sc_C
df_feature_v1["SC_O_atoms"] = sc_O
df_feature_v1["SC_total_atoms_CO"] = sc_C + sc_O
df_feature_v1["SC_bonds_CC_or_CO"] = sc_bonds
df_feature_v1["SC_total_branches"] = sc_branches

df_feature_v1["BB_C_atoms"] = bb_C
df_feature_v1["BB_hetero_atoms"] = bb_hetero
df_feature_v1["BB_CC_rotatable_bonds"] = bb_rotCC
df_feature_v1["BB_CeqC_bonds"] = bb_CeqC
df_feature_v1["BB_rings_total"] = bb_rings
df_feature_v1["BB_conjugated_units"] = bb_conj

In [7]:
# ============================================================
# 7. RDKit composition descriptors and logP features
# ============================================================

def composition_descriptors(mol):
    if mol is None:
        return {name: np.nan for name in [
            "Monomer_NumHDonors", "Monomer_NumHAcceptors", "Monomer_MolLogP",
            "Monomer_NumSpiroAtoms", "Monomer_NumSaturatedRings",
            "Monomer_NumAliphaticRings", "Monomer_NumAromaticRings",
            "Monomer_NumBridgeheadAtoms",
        ]}
    return {
        "Monomer_NumHDonors": float(Lipinski.NumHDonors(mol)),
        "Monomer_NumHAcceptors": float(Lipinski.NumHAcceptors(mol)),
        "Monomer_MolLogP": float(Crippen.MolLogP(mol)),
        "Monomer_NumSpiroAtoms": float(rdMolDescriptors.CalcNumSpiroAtoms(mol)),
        "Monomer_NumSaturatedRings": float(rdMolDescriptors.CalcNumSaturatedRings(mol)),
        "Monomer_NumAliphaticRings": float(rdMolDescriptors.CalcNumAliphaticRings(mol)),
        "Monomer_NumAromaticRings": float(rdMolDescriptors.CalcNumAromaticRings(mol)),
        "Monomer_NumBridgeheadAtoms": float(rdMolDescriptors.CalcNumBridgeheadAtoms(mol)),
    }

for column, value in composition_descriptors(mol_from_smiles_clean(monomer_smiles)).items():
    df_feature_v1[column] = float(value)


def rdkit_logp(smiles):
    mol = mol_from_smiles_clean(smiles)
    return np.nan if mol is None else float(Crippen.MolLogP(mol))


def mix_log10(values, weights):
    values = np.asarray(values, dtype=float)
    weights = np.asarray(weights, dtype=float)
    valid = np.isfinite(values) & np.isfinite(weights) & (weights > 0)
    if not valid.any():
        return np.nan
    return float(np.log10(np.sum(weights[valid] * 10.0**values[valid]) / np.sum(weights[valid])))


df_feature_v1["Backbone_MolLogP"] = rdkit_logp(backbone_smiles)
sidechain_logps = [rdkit_logp(sidechain1_smiles)]
if sidechain2_smiles is not None and str(sidechain2_smiles).strip():
    sidechain_logps.append(rdkit_logp(sidechain2_smiles))
df_feature_v1["SideChain_MolLogP"] = mix_log10(sidechain_logps, np.ones(len(sidechain_logps)))
df_feature_v1["BBminusSC_MolLogP"] = df_feature_v1["Backbone_MolLogP"] - df_feature_v1["SideChain_MolLogP"]

In [8]:
# ============================================================
# 8. Solvent logP and hydrogen-bond descriptors
# ============================================================

def solvent_hbd_hba(smiles):
    mol = mol_from_smiles_clean(smiles)
    if mol is None:
        return np.nan, np.nan
    return float(Lipinski.NumHDonors(mol)), float(Lipinski.NumHAcceptors(mol))

solvent_smiles_list = [solvent1_smiles]
solvent_weights = [float(solvent1_ratio)]
if solvent2_smiles is not None and str(solvent2_smiles).strip():
    solvent_smiles_list.append(solvent2_smiles)
    solvent_weights.append(float(solvent2_ratio))

solvent_logps = [rdkit_logp(smi) for smi in solvent_smiles_list]
solvent_hb = [solvent_hbd_hba(smi) for smi in solvent_smiles_list]
weights_array = np.asarray(solvent_weights, dtype=float)
weight_sum = weights_array.sum()

df_feature_v1["Solvent_MolLogP"] = mix_log10(solvent_logps, solvent_weights)
df_feature_v1["Solvent_NumHDonors"] = np.dot(weights_array, [v[0] for v in solvent_hb]) / weight_sum
df_feature_v1["Solvent_NumHAcceptors"] = np.dot(weights_array, [v[1] for v in solvent_hb]) / weight_sum

In [9]:
# ============================================================
# 9. RDKit 3D shape descriptors and TPSA
# ============================================================

_SHAPE_CACHE = {}

def mol3d_from_smiles_clean(smiles, seeds=(1, 7, 42, 202, 999), max_iterations=500):
    mol = mol_from_smiles_clean(smiles)
    if mol is None:
        return None
    mol = Chem.AddHs(mol)
    for seed in seeds:
        try:
            params = AllChem.ETKDGv3()
            params.randomSeed = int(seed)
            params.useRandomCoords = True
            if AllChem.EmbedMolecule(mol, params) < 0:
                continue
            if AllChem.MMFFHasAllMoleculeParams(mol):
                AllChem.MMFFOptimizeMolecule(mol, maxIters=int(max_iterations))
            else:
                AllChem.UFFOptimizeMolecule(mol, maxIters=int(max_iterations))
            return mol
        except Exception:
            continue
    return mol


def shape_tpsa_descriptors(smiles):
    if smiles is None or pd.isna(smiles) or not str(smiles).strip():
        return {"Asphericity": np.nan, "InertialShapeFactor": np.nan,
                "Eccentricity": np.nan, "TPSA": np.nan}
    key = str(smiles).strip()
    if key in _SHAPE_CACHE:
        return _SHAPE_CACHE[key].copy()
    mol2d = mol_from_smiles_clean(key)
    if mol2d is None:
        result = {"Asphericity": np.nan, "InertialShapeFactor": np.nan,
                  "Eccentricity": np.nan, "TPSA": np.nan}
        _SHAPE_CACHE[key] = result
        return result.copy()
    try:
        tpsa = float(rdMolDescriptors.CalcTPSA(mol2d))
    except Exception:
        tpsa = np.nan
    asphericity = inertial_shape_factor = eccentricity = np.nan
    try:
        mol3d = mol3d_from_smiles_clean(key)
        if mol3d is not None and mol3d.GetNumConformers() > 0:
            asphericity = float(rdMolDescriptors.CalcAsphericity(mol3d))
            inertial_shape_factor = float(rdMolDescriptors.CalcInertialShapeFactor(mol3d))
            eccentricity = float(rdMolDescriptors.CalcEccentricity(mol3d))
    except Exception:
        pass
    result = {"Asphericity": asphericity,
              "InertialShapeFactor": inertial_shape_factor,
              "Eccentricity": eccentricity, "TPSA": tpsa}
    _SHAPE_CACHE[key] = result
    return result.copy()


def weighted_linear_descriptor_mix(descriptor_dicts, weights):
    weights = np.asarray(weights, dtype=float)
    output = {}
    for key in descriptor_dicts[0]:
        values = np.asarray([d[key] for d in descriptor_dicts], dtype=float)
        valid = np.isfinite(values) & np.isfinite(weights) & (weights > 0)
        output[key] = np.nan if not valid.any() else float(np.dot(weights[valid], values[valid]) / weights[valid].sum())
    return output

monomer_shape = shape_tpsa_descriptors(monomer_smiles)
backbone_shape = shape_tpsa_descriptors(backbone_smiles)
sidechain_shape_inputs = [shape_tpsa_descriptors(sidechain1_smiles)]
if sidechain2_smiles is not None and str(sidechain2_smiles).strip():
    sidechain_shape_inputs.append(shape_tpsa_descriptors(sidechain2_smiles))
sidechain_shape = weighted_linear_descriptor_mix(sidechain_shape_inputs, np.ones(len(sidechain_shape_inputs)))
solvent_shape_inputs = [shape_tpsa_descriptors(smi) for smi in solvent_smiles_list]
solvent_shape = weighted_linear_descriptor_mix(solvent_shape_inputs, solvent_weights)

for prefix, descriptor in [
    ("Monomer", monomer_shape), ("Backbone", backbone_shape),
    ("SideChain", sidechain_shape), ("Solvent", solvent_shape),
]:
    for name, value in descriptor.items():
        df_feature_v1[f"{prefix}_{name}"] = float(value)

In [10]:
# ============================================================
# 10. FASTSOLV features
# ============================================================
# FASTSOLV receives solvent SMILES, solute SMILES, and temperature in K.
# Solvent mixtures and two side-chain types are mixed in linear
# solubility space. No PM7-specific overrides are applied.

from fastsolv import fastsolv


def extract_fastsolv_prediction(output, expected_length):
    if isinstance(output, pd.Series):
        prediction = output.copy()
    elif isinstance(output, (np.ndarray, list, tuple)):
        prediction = pd.Series(output)
    elif isinstance(output, pd.DataFrame):
        preferred = ["predicted_logS", "predicted_logs", "logS", "logs", "prediction", "pred"]
        lower_to_original = {str(c).lower(): c for c in output.columns}
        selected = next((lower_to_original[n.lower()] for n in preferred if n.lower() in lower_to_original), None)
        if selected is None:
            numeric_columns = output.select_dtypes(include=[np.number]).columns.tolist()
            candidates = [c for c in numeric_columns
                          if re.search(r"(log.?s|solub|pred)", str(c), flags=re.I)
                          and not re.search(r"(std|stdev|sigma|uncert|variance|var)", str(c), flags=re.I)]
            selected = candidates[0] if candidates else (numeric_columns[0] if numeric_columns else None)
        if selected is None:
            raise ValueError("FASTSOLV output does not contain a numerical prediction column.")
        prediction = output[selected].copy()
    else:
        raise TypeError(f"Unexpected FASTSOLV output type: {type(output)}")

    prediction = pd.to_numeric(prediction, errors="coerce").reset_index(drop=True)
    if len(prediction) != expected_length:
        raise ValueError(f"FASTSOLV returned {len(prediction)} predictions for {expected_length} queries.")
    return prediction


temperature_K = float(df_feature_v1.loc[0, "Temperature"])
solute_representations = {
    "monomer": clean_smiles_remove_dummy(monomer_smiles),
    "backbone": clean_smiles_remove_dummy(backbone_smiles),
    "sidechain1": clean_smiles_remove_dummy(sidechain1_smiles),
}
if sidechain2_smiles is not None and str(sidechain2_smiles).strip():
    solute_representations["sidechain2"] = clean_smiles_remove_dummy(sidechain2_smiles)

clean_solvents = [clean_smiles_remove_dummy(smi) for smi in solvent_smiles_list]
query_records = []
for solute_label, solute_smi in solute_representations.items():
    for solvent_index, solvent_smi in enumerate(clean_solvents):
        query_records.append({
            "solute_label": solute_label,
            "solvent_index": solvent_index,
            "solvent_smiles": solvent_smi,
            "solute_smiles": solute_smi,
            "temperature": temperature_K,
        })

query_table = pd.DataFrame(query_records)
fastsolv_input = query_table[["solvent_smiles", "solute_smiles", "temperature"]].copy()
fastsolv_output = fastsolv(fastsolv_input)
query_table["predicted_logS"] = extract_fastsolv_prediction(fastsolv_output, len(query_table)).to_numpy()


def mixed_fastsolv_for_solute(solute_label):
    rows = query_table.loc[query_table["solute_label"].eq(solute_label)].sort_values("solvent_index")
    return mix_log10(rows["predicted_logS"].to_numpy(dtype=float), solvent_weights)


df_feature_v1["Monomer_fastsolv"] = mixed_fastsolv_for_solute("monomer")
df_feature_v1["Backbone_fastsolv"] = mixed_fastsolv_for_solute("backbone")
sidechain_fastsolv_values = [mixed_fastsolv_for_solute("sidechain1")]
if "sidechain2" in solute_representations:
    sidechain_fastsolv_values.append(mixed_fastsolv_for_solute("sidechain2"))
df_feature_v1["SideChain_fastsolv"] = mix_log10(sidechain_fastsolv_values, np.ones(len(sidechain_fastsolv_values)))

100%|████████████████████████████████████████████████████████████████████████████████████| 5/5 [00:00<00:00,  5.34it/s]
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores


Predicting DataLoader 0: 100%|███████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 72.34it/s]


In [11]:
# ============================================================
# 11. Fixed family one-hot features and COSMO-RS input
# ============================================================

for category in FAMILY_CATEGORIES:
    df_feature_v1[f"family_{category}"] = int(family == category)

df_feature_v1["dG_solv_cosmo"] = float(dG_solv_cosmo)

In [12]:
# ============================================================
# 12. Molecular-weight normalization
# ============================================================
# Divide selected SC_, BB_, and monomer-count descriptors by
# MolWt_monomer / 1000, then replace the original columns.

molwt_denominator = pd.to_numeric(df_feature_v1["MolWt_monomer"], errors="coerce") / 1000.0
if molwt_denominator.isna().any() or (molwt_denominator == 0).any():
    raise ValueError("MolWt_monomer is missing or zero.")

explicit_monomer_columns = [
    "Monomer_NumHDonors", "Monomer_NumHAcceptors",
    "Monomer_NumSpiroAtoms", "Monomer_NumSaturatedRings",
    "Monomer_NumAliphaticRings", "Monomer_NumAromaticRings",
    "Monomer_NumBridgeheadAtoms",
]
sc_columns = [c for c in df_feature_v1.columns if str(c).startswith("SC_")]
bb_columns = [c for c in df_feature_v1.columns if str(c).startswith("BB_")]
columns_to_normalize = [c for c in df_feature_v1.columns
                        if c in set(sc_columns + bb_columns + explicit_monomer_columns)]

for column in columns_to_normalize:
    normalized_column = f"{column}_Norm_MolWt"
    df_feature_v1[normalized_column] = pd.to_numeric(df_feature_v1[column], errors="coerce") / molwt_denominator
    df_feature_v1.drop(columns=[column], inplace=True)

In [13]:
# ============================================================
# 13. Print the final 77 features
# ============================================================

missing_features = [c for c in FINAL_FEATURE_COLUMNS if c not in df_feature_v1.columns]
if missing_features:
    raise KeyError("The following final features were not generated:\n" + "\n".join(missing_features))

final_features = df_feature_v1.loc[:, FINAL_FEATURE_COLUMNS].copy()
if final_features.shape != (1, 77):
    raise ValueError(f"Expected a (1, 77) feature table, obtained {final_features.shape}.")

print(f"Generated {final_features.shape[1]} features for one data point:\n")
for feature_name, feature_value in final_features.iloc[0].items():
    print(f"{feature_name}\t{feature_value}")

Generated 77 features for one data point:

MW_Mn	30000.0
MW_Mw	52500.0
PDI	1.75
MolWt_monomer	801.1619999999992
Temperature	298.15
R_atoms_C	4.25
R_atom_all	4.4
R_bond_1	10.0
R_bond_2	20.0
R_ring_1	2.0
R_ring_2	2.0
R_ring_3	20.0
R_ring_4	2.0
R_ring_5	2.0
R_ring_6	20.0
SC_n_branch_points_Norm_MolWt	0.0
SC_atoms_after_branching_CO_Norm_MolWt	54.9202283682951
SC_n_units_Norm_MolWt	4.992748033481373
SC_atoms_before_branching_CO_Norm_MolWt	0.0
SC_atoms_before_plus_after_CO_Norm_MolWt	54.9202283682951
SC_C_atoms_Norm_MolWt	42.43835828459167
SC_O_atoms_Norm_MolWt	12.481870083703432
SC_total_atoms_CO_Norm_MolWt	54.9202283682951
SC_bonds_CC_or_CO_Norm_MolWt	49.92748033481373
SC_total_branches_Norm_MolWt	4.992748033481373
BB_C_atoms_Norm_MolWt	9.985496066962746
BB_hetero_atoms_Norm_MolWt	2.4963740167406865
BB_CC_rotatable_bonds_Norm_MolWt	2.4963740167406865
BB_CeqC_bonds_Norm_MolWt	4.992748033481373
BB_rings_total_Norm_MolWt	2.4963740167406865
BB_conjugated_units_Norm_MolWt	2.4963740167406865
Mo

In [14]:
# ============================================================
# 14. Validate the generated features against a reference row
# ============================================================
# The first number (88) is treated as the data-point ID.
# The remaining 77 values are assumed to follow FINAL_FEATURE_COLUMNS
# in exactly the same order as final_features.

reference_row = """
88 30000  52500  1.75  801.162  298.15  4.25  4.4  10  20  2  2  20  2  2  20  0
54.92022837  4.992748033  0  54.92022837  42.43835828  12.48187008  54.92022837
49.92748033  4.992748033  9.985496067  2.496374017  2.496374017  4.992748033
2.496374017  2.496374017  0  14.9782441  10.6324  0  0  0  2.496374017  0
3.511  3.456020183  0.054979817  1.9864  0  0  0.076906374  6.75026E-05
0.781695127  92.3  0.426097335  0.001298536  0.967166031  36.92
0.389574773  0.004489819  0.941350287  13.845  0.222014209  0.003324492
0.854417531  0  -0.141165122  0.288954556  1.105718915
1  0  0  0  0  0  0  0  0  0  0  -45.062557  100
"""

# Tolerances account for the decimal rounding in the supplied reference row.
ATOL = 1e-6
RTOL = 1e-6

reference_numbers = np.fromstring(reference_row, sep=" ", dtype=float)

if reference_numbers.size != 78:
    raise ValueError(
        f"Expected 77 numbers (one ID + 77 features), "
        f"found {reference_numbers.size}."
    )

data_point_id = int(reference_numbers[0])
reference_values = reference_numbers[1:]

if final_features.shape != (1, 77):
    raise ValueError(
        f"Expected final_features shape (1, 77), "
        f"found {final_features.shape}."
    )

if list(final_features.columns) != list(FINAL_FEATURE_COLUMNS):
    raise ValueError(
        "final_features columns are not in FINAL_FEATURE_COLUMNS order."
    )

generated_values = (
    final_features.iloc[0]
    .apply(pd.to_numeric, errors="coerce")
    .to_numpy(dtype=float)
)

comparison = pd.DataFrame({
    "feature": FINAL_FEATURE_COLUMNS,
    "generated": generated_values,
    "reference": reference_values,
})

comparison["abs_diff"] = np.abs(
    comparison["generated"] - comparison["reference"]
)

comparison["rel_diff"] = comparison["abs_diff"] / np.maximum(
    np.abs(comparison["reference"]),
    ATOL,
)

comparison["match"] = np.isclose(
    comparison["generated"],
    comparison["reference"],
    atol=ATOL,
    rtol=RTOL,
    equal_nan=True,
)

n_match = int(comparison["match"].sum())
n_total = len(comparison)

print(f"Reference data-point ID: {data_point_id}")
print(f"Generated feature count: {generated_values.size}")
print(f"Reference feature count: {reference_values.size}")
print(f"Matched within tolerance: {n_match}/{n_total}")
print(
    "Maximum absolute difference: "
    f"{comparison['abs_diff'].max():.12g}"
)

if n_match == n_total:
    print(
        "PASS: all 77 generated features match the "
        "reference row within tolerance."
    )
else:
    print(
        f"FAIL: {n_total - n_match} feature(s) differ "
        "from the reference row."
    )
    print("\nMismatched features:")
    display(
        comparison.loc[~comparison["match"]]
        .reset_index(drop=True)
    )

print("\nFull feature-by-feature comparison:")
display(comparison)

Reference data-point ID: 88
Generated feature count: 77
Reference feature count: 77
Matched within tolerance: 77/77
Maximum absolute difference: 4.81372808281e-09
PASS: all 77 generated features match the reference row within tolerance.

Full feature-by-feature comparison:


,feature,generated,reference,abs_diff,rel_diff,match
0,MW_Mn,30000.000000,30000.000000,0.000000e+00,0.000000e+00,True
1,MW_Mw,52500.000000,52500.000000,0.000000e+00,0.000000e+00,True
2,PDI,1.750000,1.750000,0.000000e+00,0.000000e+00,True
3,MolWt_monomer,801.162000,801.162000,7.958079e-13,9.933170e-16,True
4,Temperature,298.150000,298.150000,0.000000e+00,0.000000e+00,True
...,...,...,...,...,...,...
72,family_Polyfluorene,0.000000,0.000000,0.000000e+00,0.000000e+00,True
73,family_IDT,0.000000,0.000000,0.000000e+00,0.000000e+00,True
74,family_Ladder,0.000000,0.000000,0.000000e+00,0.000000e+00,True
75,dG_solv_cosmo,-45.062557,-45.062557,0.000000e+00,0.000000e+00,True
